# 04 - Entrenamiento de RT-DETR para detección de humo y fuego

RT-DETR es un detector basado en transformers con NMS-free y velocidad de
tiempo real. Se entrena con la misma API de Ultralytics que el baseline YOLOv8n,
así que este notebook reusa el pipeline del notebook 02.

Las métricas finales y las figuras del reporte (matriz de confusión, curvas
P/R/F1, `metrics_summary.csv`) se calculan sobre el split de **test**;
`results.csv`/`results.png` son la historia del entrenamiento, cuyas métricas
por época se miden sobre val, que es también el split con el que Ultralytics
elige `best.pt`.

## Dónde correrlo

El notebook detecta el entorno y se adapta solo: corre en **Colab**, en **Kaggle
Notebooks** o en una **GPU local**.

RT-DETR-L cuesta unas 12 veces más que YOLOv8n (108 vs 8.7 GFLOPs): en una T4 de
Kaggle da unos 24 min por época, así que las 30 épocas son del orden de 12 h. Son
las mismas 30 épocas que YOLOv8n y Faster R-CNN, para que la comparación no
premie al modelo que entrenó más tiempo. Eso excede el plan gratuito de Colab y
roza el límite de 12 h por sesión de Kaggle, así que conviene contar con **dos
sesiones**: el entrenamiento reanuda desde `last.pt` y `train_time_min` se
acumula entre sesiones. Sobre la cuota semanal de Kaggle son 12 de las 30 h.

Hace falta una GPU de **16 GB**. A `imgsz 640` y `batch 8` el modelo ocupa
8-10 GB, así que las placas de 2-4 GB quedan afuera: bajar `imgsz` o `batch`
para que entren rompería la comparabilidad con el resto de la tabla.

## Ampliar las épocas de una corrida ya empezada

Si `epochs` sube en el YAML con una corrida a medio hacer, alcanza con volver a
ejecutar el notebook: la celda de entrenamiento reescribe el total dentro de
`last.pt` y reanuda. Hace falta porque con `resume=True` Ultralytics ignora el
`epochs` que se le pase y usa el que quedó guardado en el checkpoint.

Solo funciona **mientras la corrida siga viva**: al terminar todas sus épocas,
Ultralytics le pasa `strip_optimizer` a `last.pt` y le saca el optimizador y el
EMA, con lo que ya no hay nada que reanudar. Después de eso, ampliar el total
implica entrenar de cero. Cuanto antes se amplíe, mejor: el salto de learning
rate y la reapertura del mosaico son menores cuanto más lejos está el final.

## Reevaluar una corrida terminada sin reentrenar

Con la corrida completa montada como input (**Add Input > Your Work**, la
versión Quick Save), volver a ejecutar el notebook entero no reentrena nada:
`training_completed.txt` hace que la celda de entrenamiento se saltee y se pasa
directo a la evaluación de `best.pt` sobre test. Sirve para regenerar las
métricas si cambió la evaluación.

### Kaggle en varias sesiones

Kaggle da 30 h semanales de GPU, y el dataset D-Fire ya está publicado ahí, así
que no hay que descargarlo.

1. Notebook nuevo, panel derecho **Session options > Accelerator > GPU T4 x2**
   e **Internet > On**. Ambas opciones aparecen recién cuando la cuenta está
   verificada por teléfono. No elegir la P100: es `sm_60` y el PyTorch que
   trae la imagen actual de Kaggle solo incluye kernels desde `sm_70`, así
   que `torch.cuda.is_available()` da `True` pero la corrida muere en el
   primer forward. De las dos T4 se usa una sola: para aprovechar ambas
   Ultralytics necesita DDP, y dentro de un notebook eso es frágil.
2. **Add Input > Datasets** y buscar `sayedgamal99/smoke-fire-detection-yolo`.
3. Correr todo. Antes de las 12 h, **Save Version > Quick Save**: guarda
   `/kaggle/working` de esta sesión como output de la versión, con los pesos
   adentro. Ojo que **Save & Run All** no sirve acá, porque reejecuta el
   notebook desde cero en un contenedor limpio y tira la corrida.
4. Para cada sesión siguiente, **Add Input > Your Work** y elegir la última
   versión guardada. La celda de recuperación copia la corrida de vuelta a
   `/kaggle/working` y el entrenamiento sigue donde quedó.
5. Al terminar, la última celda arma un `.zip` con los artefactos y `best.pt`
   para bajar, commitear en el repo y subir a Drive, y así seguir en Colab con
   el notebook 05.

In [1]:
# ============================================================
# Setup general del entorno
# ============================================================

from pathlib import Path
import os
import sys
import random
import shutil
import time
import yaml

SEED = 42
random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
# KAGGLE_KERNEL_RUN_TYPE lo define el runtime de Kaggle y no existe si alguien
# instala el paquete `kaggle` en otra máquina, a diferencia de /kaggle o de la
# variable KAGGLE_URL_BASE que trae la librería.
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB:
    ENV = "colab"
elif IN_KAGGLE:
    ENV = "kaggle"
else:
    ENV = "local"

print("Entorno:", ENV)
print("Directorio actual:", Path.cwd())

Entorno: kaggle
Directorio actual: /kaggle/working


In [2]:
# ============================================================
# Instalación de dependencias
# ============================================================

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego"
REPO_NAME = "VpC2---Deteccion-de-humo-y-fuego"
# Rama del repositorio desde la que se clona y a la que se commitean los
# resultados. Mientras el PR este abierto tiene que apuntar a la rama del PR;
# una vez mergeado, cambiar a "main".
REPO_BRANCH = "feat/modelos-adicionales-deteccion"

RAW_REQUIREMENTS = (
    "https://raw.githubusercontent.com/Gabriela-Sol/"
    f"{REPO_NAME}/{REPO_BRANCH}/requirements.txt"
)

if IN_COLAB:
    !pip install -q -r {RAW_REQUIREMENTS}
elif IN_KAGGLE:
    # Solo ultralytics: requirements.txt lista `torch` sin pinear la build de
    # CUDA, y dejar que pip lo resuelva en Kaggle reemplaza el torch
    # preinstalado (que sí viene compilado contra el CUDA de la imagen) por uno
    # cualquiera de PyPI. El resto de las dependencias ya está en la imagen.
    !pip install -q "ultralytics>=8.3,<8.5"

print("Dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.3 MB/s eta 0:00:00
Dependencias instaladas.


In [3]:
# ============================================================
# Verificación de GPU
# ============================================================

import torch

print("CUDA disponible:", torch.cuda.is_available())

# RT-DETR-L a imgsz 640 y batch 8 ocupa del orden de 8-10 GB. Por debajo de eso
# la corrida muere con OOM recién al empezar la primera época, así que conviene
# avisar acá.
VRAM_MINIMA_GB = 8.0

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_NAME = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {DEVICE_NAME} ({vram_gb:.1f} GB)")
    if vram_gb < VRAM_MINIMA_GB:
        print(
            f"ATENCION: {vram_gb:.1f} GB de VRAM son pocos para rtdetr-l a "
            f"imgsz 640 / batch 8. Esperar OOM salvo que se bajen ambos, lo que "
            f"rompe la comparabilidad con el resto de la tabla."
        )
else:
    DEVICE = torch.device("cpu")
    DEVICE_NAME = "cpu"
    if IN_COLAB:
        # Cortar acá y no avisar nomás: en CPU la corrida no termina nunca y el
        # usuario se enteraría recién dentro de varias horas.
        raise RuntimeError(
            "No se detectó GPU. RT-DETR en CPU es inviable para 20 épocas. "
            "Activar Entorno de ejecución > Cambiar tipo de entorno > GPU."
        )
    if IN_KAGGLE:
        raise RuntimeError(
            "No se detectó GPU. RT-DETR en CPU es inviable para 20 épocas. "
            "Activar Settings > Accelerator > GPU P100."
        )
    print("Sin GPU y fuera de Colab: se sigue en CPU, solo sirve para pruebas cortas.")

print("Device:", DEVICE)

CUDA disponible: True
GPU: Tesla T4 (14.6 GB)
Device: cuda


In [4]:
# ============================================================
# Almacenamiento de trabajo (Drive en Colab)
# ============================================================
# WORK_DIR es la raíz escribible del entorno: de ahí salen el clon del repo, el
# YAML del dataset y, salvo en Colab, las corridas de Ultralytics.

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/VCII_DFire")
    DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
    DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
    print("Carpeta principal en Drive:", DRIVE_PROJECT_DIR)
    print("Carpeta de corridas:", DRIVE_RUNS_DIR)

    WORK_DIR = Path("/content")
elif IN_KAGGLE:
    # Único directorio escribible que Kaggle preserva al hacer Quick Save, y por
    # lo tanto el único desde el que se puede reanudar en otra sesión.
    WORK_DIR = Path("/kaggle/working")
else:
    WORK_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

print("WORK_DIR:", WORK_DIR)

WORK_DIR: /kaggle/working


In [5]:
# ============================================================
# Clonado o actualización del repositorio
# ============================================================

if ENV == "local":
    # Ya estamos dentro del repo: no hay nada que clonar.
    PROJECT_DIR = WORK_DIR
else:
    PROJECT_DIR = WORK_DIR / REPO_NAME

    if PROJECT_DIR.exists():
        print("El repositorio ya existe. Actualizando...")
        %cd {PROJECT_DIR}
        !git checkout {REPO_BRANCH}
        !git pull origin {REPO_BRANCH}
    else:
        print("Clonando repositorio...")
        %cd {WORK_DIR}
        !git clone -b {REPO_BRANCH} {REPO_URL}.git
        %cd {PROJECT_DIR}

# Necesario para que `import src...` funcione.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)
print("Contenido del proyecto:", os.listdir(PROJECT_DIR))

Clonando repositorio...
/kaggle/working
Cloning into 'VpC2---Deteccion-de-humo-y-fuego'...
remote: Enumerating objects: 835, done.
remote: Counting objects: 100% (208/208), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 835 (delta 139), reused 129 (delta 85), pack-reused 627 (from 3)
Receiving objects: 100% (835/835), 46.16 MiB | 34.06 MiB/s, done.
Resolving deltas: 100% (438/438), done.
/kaggle/working/VpC2---Deteccion-de-humo-y-fuego
PROJECT_DIR: /kaggle/working/VpC2---Deteccion-de-humo-y-fuego
Contenido del proyecto: ['.gitignore', 'data', 'requirements.txt', 'tests', 'pytest.ini', 'notebooks', 'reports', 'configs', 'src', '.git', 'README.md', 'demo']


In [7]:
# ============================================================
# Carga de configuración del experimento
# ============================================================

EXPERIMENT_CONFIG_PATH = PROJECT_DIR / "configs" / "experiments" / "rtdetr_l.yaml"

with open(EXPERIMENT_CONFIG_PATH, "r", encoding="utf-8") as file:
    experiment_config = yaml.safe_load(file)

experiment_name = experiment_config["experiment"]["name"]
model_name = experiment_config["experiment"]["model"]
training_cfg = experiment_config["training"]

# `output.project` del YAML apunta a Drive, que solo existe en Colab. La ruta
# efectiva se resuelve acá en vez de editar `experiment_config`, para que el
# experiment_config_used.yaml salga igual en los tres entornos y los diffs entre
# corridas no muestren cambios de ruta que no son hiperparámetros.
if IN_COLAB:
    RUNS_DIR = Path(experiment_config["output"]["project"])
else:
    RUNS_DIR = WORK_DIR / "runs"

print("Experimento:", experiment_name)
print("Modelo:", model_name, "| épocas:", training_cfg["epochs"])
print("Corridas en:", RUNS_DIR)

Experimento: rtdetr_l
Modelo: rtdetr-l.pt | épocas: 30
Corridas en: /kaggle/working/runs


In [9]:
# ============================================================
# Dataset y YAML para Ultralytics
# ============================================================

DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"


def find_yolo_dataset_dir(root: Path) -> Path:
    for candidate in [root] + [p for p in root.rglob("*") if p.is_dir()]:
        if all(
            (candidate / split / kind).exists()
            for split in ["train", "val"]
            for kind in ["images", "labels"]
        ):
            return candidate
    raise FileNotFoundError("No se encontró una estructura YOLO válida.")


DATA_DIR = None

if IN_KAGGLE:
    # El dataset ya está publicado en Kaggle, así que si se agregó con
    # Add Input > Datasets se monta en /kaggle/input y no hay nada que bajar.
    for mount in sorted(Path("/kaggle/input").glob("*")):
        if not mount.is_dir():
            continue
        try:
            DATA_DIR = find_yolo_dataset_dir(mount)
        except FileNotFoundError:
            continue
        print("Dataset montado desde los inputs del notebook:", mount.name)
        break
    if DATA_DIR is None:
        print(
            f"No se encontró el dataset en /kaggle/input. Agregarlo con "
            f"Add Input > Datasets > {DATASET_ID} para evitar la descarga."
        )

if DATA_DIR is None:
    import kagglehub

    DATA_DIR = find_yolo_dataset_dir(Path(kagglehub.dataset_download(DATASET_ID)))

DFIRE_YAML = WORK_DIR / "dfire_dataset.yaml"

with open(DFIRE_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        {
            "path": str(DATA_DIR),
            "train": "train/images",
            "val": "val/images",
            "test": "test/images",
            "nc": 2,
            "names": {0: "smoke", 1: "fire"},
        },
        file,
        sort_keys=False,
        allow_unicode=True,
    )

print("Dataset:", DATA_DIR)
print(DFIRE_YAML.read_text())

if IN_KAGGLE:
    # /kaggle/input es read-only: Ultralytics detecta que no puede escribir el
    # labels.cache al lado de las etiquetas, avisa y sigue. El costo es volver a
    # escanear las ~17k etiquetas en cada sesión, un par de minutos.
    print("Nota: el aviso 'cache directory ... is not writeable' es esperado.")

Dataset montado desde los inputs del notebook: datasets
Dataset: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data
path: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data
train: train/images
val: val/images
test: test/images
nc: 2
names:
  0: smoke
  1: fire

Nota: el aviso 'cache directory ... is not writeable' es esperado.


In [13]:
# ============================================================
# Recuperar la corrida de una sesión anterior (Kaggle)
# ============================================================
# Kaggle borra /kaggle/working al cerrar la sesión, pero Quick Save lo guarda
# como output de la versión. Si ese output se volvió a montar con
# Add Input > Your Work, la corrida se copia de vuelta acá y la celda de
# entrenamiento reanuda desde last.pt (o se saltea, si ya terminó) en lugar de
# arrancar de cero.
#
# La profundidad a la que queda montado el output no es fija, así que la
# corrida se busca a cualquier profundidad bajo /kaggle/input: `*/` saltea el
# nivel del slug y `**` matchea cero o más directorios, igual que en
# src/data/kaggle_inputs.py.
#
# En Colab no hace falta: la corrida vive en Drive, que persiste entre sesiones.

import stat

experiment_dir = RUNS_DIR / experiment_name


def _pesos_de(corrida: Path) -> list[Path]:
    weights_dir = corrida / "weights"
    return sorted(weights_dir.glob("*.pt")) if weights_dir.is_dir() else []


if IN_KAGGLE and not _pesos_de(experiment_dir):
    previas = {
        pesos.parent.parent
        for pesos in Path("/kaggle/input").glob(f"*/**/{experiment_name}/weights/*.pt")
    }

    if previas:
        # Elegimos por fecha de modificación del peso más reciente para asegurar
        # la versión más nueva. El orden alfabético es incorrecto para números
        # con diferente cantidad de dígitos (e.g. mi-version-9 vs mi-version-10).
        origen = max(previas, key=lambda p: max(f.stat().st_mtime for f in _pesos_de(p)))
        experiment_dir.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(origen, experiment_dir, dirs_exist_ok=True)

        # copytree usa copy2, que preserva los permisos: los archivos vienen de
        # /kaggle/input, que es read-only, y Ultralytics necesita sobreescribir
        # last.pt en cada época. Sin esto el resume muere con PermissionError
        # recién al cerrar la primera época.
        for ruta in experiment_dir.rglob("*"):
            ruta.chmod(ruta.stat().st_mode | stat.S_IWUSR)

        print("Corrida anterior recuperada desde:", origen)
    else:
        print("No hay corrida anterior montada: se entrena desde cero.")
        # Listado corto de lo montado para diagnosticar por qué no se encontró:
        # el output tiene que traer runs/<experimento>/weights/*.pt en alguna
        # parte. Si el slug del notebook aparece vacío o sin runs/, la versión
        # montada se guardó sin output (o antes de entrenar): en el panel de
        # inputs, elegir la versión del Quick Save que sí tiene los archivos.
        print("Inputs montados en /kaggle/input:")
        for mount in sorted(Path("/kaggle/input").iterdir()):
            print("  ", mount.name)
            if mount.is_dir():
                for hijo in sorted(mount.iterdir())[:10]:
                    print("      ", hijo.name + ("/" if hijo.is_dir() else ""))

print("experiment_dir:", experiment_dir)

Corrida anterior recuperada desde: /kaggle/input/notebooks/marcoslund/notebook-rt-detr/runs/rtdetr_l
experiment_dir: /kaggle/working/runs/rtdetr_l


In [14]:
# ============================================================
# Entrenamiento con RT-DETR
# ============================================================

from ultralytics import RTDETR

RUNS_DIR.mkdir(parents=True, exist_ok=True)

# metrics_summary.csv reporta train_time_min, y una corrida partida en dos
# sesiones de Kaggle mediría solo la última: el número quedaría por debajo del
# costo real y sería incomparable contra las otras filas de la tabla. Así que el
# tiempo se acumula en un archivo dentro de la corrida.
#
# La columna `time` del results.csv de Ultralytics no sirve para esto: se
# reinicia en cada resume porque arranca de un train_time_start nuevo.
TRAIN_TIME_FILE = experiment_dir / "train_time_min.txt"

# Marca explícita de "esta corrida ya terminó". Hace falta porque al completar el
# entrenamiento Ultralytics le pasa strip_optimizer a last.pt, que deja
# epoch = -1, y entonces resume=True no reanuda nada: corta con AssertionError.
# Sin la marca, volver a ejecutar el notebook entero sobre una corrida terminada
# (algo normal al retomar en otra sesión) rompe en esta celda.
COMPLETED_FILE = experiment_dir / "training_completed.txt"

minutos_previos = (
    float(TRAIN_TIME_FILE.read_text()) if TRAIN_TIME_FILE.exists() else 0.0
)
start_time = time.time()


def registrar_tiempo_acumulado(trainer):
    """Persiste el tiempo al cerrar cada época.

    Se escribe por época y no al final del train() porque el caso que importa es
    justamente el que no llega al final: cuando Kaggle corta la sesión a las
    12 h, la celda nunca retorna y un cálculo post-train perdería esas horas.
    """
    TRAIN_TIME_FILE.write_text(
        f"{minutos_previos + (time.time() - start_time) / 60:.4f}"
    )


def reescribir_epocas_del_checkpoint(last_pt: Path, epochs: int) -> None:
    """Ajusta el total de épocas guardado dentro de `last.pt`.

    Con `resume=True` Ultralytics toma todos los hiperparámetros del checkpoint y
    solo deja pisar imgsz, batch, device y close_mosaic, así que ampliar `epochs`
    en el YAML no tiene ningún efecto por sí solo: hay que reescribirlo adentro
    del checkpoint. Al reanudar, el scheduler se rearma con el total nuevo, de
    modo que las épocas que faltan siguen la curva de LR que habrían tenido en
    una corrida de `epochs` épocas.
    """
    checkpoint = torch.load(last_pt, map_location="cpu", weights_only=False)
    epocas_ckpt = checkpoint["train_args"]["epochs"]

    if checkpoint.get("epoch", -1) < 0:
        # strip_optimizer ya pasó por este archivo: la corrida terminó y no
        # quedan ni optimizador ni EMA, así que no hay nada que reanudar. Ampliar
        # las épocas solo es posible mientras la corrida sigue viva.
        raise RuntimeError(
            f"{last_pt} es de una corrida ya terminada (sin optimizador ni EMA): "
            f"no se puede reanudar para llegar a {epochs} épocas. Opciones: "
            f"dejar el YAML en las {epocas_ckpt} épocas que se entrenaron, o "
            f"borrar la corrida y entrenar de cero con el total nuevo."
        )

    if epocas_ckpt == epochs:
        return

    checkpoint["train_args"]["epochs"] = epochs
    # Se escribe a un temporal y recién después se reemplaza: si la sesión se
    # corta en medio del torch.save, un last.pt truncado se lleva puesta toda la
    # corrida.
    temporal = last_pt.with_suffix(".pt.tmp")
    torch.save(checkpoint, temporal)
    os.replace(temporal, last_pt)

    print(
        f"El checkpoint venía de una corrida de {epocas_ckpt} épocas: se "
        f"reescribe el total a {epochs} y se reanuda desde la época "
        f"{checkpoint['epoch'] + 2}."
    )


if COMPLETED_FILE.exists():
    print("El entrenamiento de esta corrida ya está completo: no se reentrena.")
    print("Para forzar uno nuevo, borrar", COMPLETED_FILE)
else:
    last_pt = experiment_dir / "weights" / "last.pt"

    if last_pt.exists():
        print("Checkpoint encontrado, reanudando entrenamiento.")
        reescribir_epocas_del_checkpoint(last_pt, training_cfg["epochs"])
        model = RTDETR(str(last_pt))
        train_kwargs = {"resume": True}
    else:
        print("Entrenamiento desde los pesos preentrenados.")
        model = RTDETR(model_name)
        train_kwargs = {
            "data": str(DFIRE_YAML),
            "epochs": training_cfg["epochs"],
            "imgsz": training_cfg["imgsz"],
            "batch": training_cfg["batch"],
            "patience": training_cfg["patience"],
            "optimizer": training_cfg["optimizer"],
            "lr0": training_cfg["lr0"],
            "seed": training_cfg["seed"],
            "project": str(RUNS_DIR),
            "name": experiment_name,
            "exist_ok": True,
            "plots": True,
        }

    model.add_callback("on_fit_epoch_end", registrar_tiempo_acumulado)
    results = model.train(**train_kwargs)

    # Solo se llega acá si train() volvió sin excepción: agotó las épocas o cortó
    # por patience.
    COMPLETED_FILE.write_text("ok\n")
    print(f"Esta sesión: {(time.time() - start_time) / 60:.1f} min")

train_time_min = (
    float(TRAIN_TIME_FILE.read_text()) if TRAIN_TIME_FILE.exists() else minutos_previos
)

print(f"Tiempo acumulado de la corrida: {train_time_min:.1f} min")
print("Resultados en:", experiment_dir)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
El entrenamiento de esta corrida ya está completo: no se reentrena.
Para forzar uno nuevo, borrar /kaggle/working/runs/rtdetr_l/training_completed.txt
Tiempo acumulado de la corrida: 562.9 min
Resultados en: /kaggle/working/runs/rtdetr_l


In [15]:
# ============================================================
# Evaluación final sobre test
# ============================================================
# Las métricas finales se calculan sobre el split de test, que no participó ni
# del entrenamiento ni de la selección de best.pt (esa se hace sobre val).

best_weights = experiment_dir / "weights" / "best.pt"
model = RTDETR(str(best_weights))

metrics = model.val(data=str(DFIRE_YAML), split="test", plots=True)

# val() guarda sus figuras (matriz de confusión, curvas P/R/F1) en un
# directorio propio, distinto del de entrenamiento: la celda siguiente copia
# desde acá las figuras que corresponden a test.
EVAL_DIR = Path(metrics.save_dir)

print("mAP50   :", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Figuras de la evaluación en:", EVAL_DIR)

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary: 315 layers, 31,987,850 parameters, 0 gradients, 105.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 12.4±12.6 MB/s, size: 91.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/labels... 4295 images, 2005 backgrounds, 15 corrupt: 100% ━━━━━━━━━━━━ 4306/4306 232.8it/s 18.5s0.0s
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images/WEB10769.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0297]
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images/WEB10775.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0156]
val: /kaggle/input/datasets/sayedgamal99

In [18]:
# ============================================================
# Copiar los artefactos de Ultralytics al repositorio
# ============================================================
# Dos orígenes: results.csv/results.png son la historia del entrenamiento
# (métricas por época sobre val) y viven en experiment_dir; la matriz de
# confusión y las curvas P/R/F1 salen de la evaluación sobre test, así que se
# copian desde EVAL_DIR y no desde las figuras que dejó el entrenamiento.

REPORTS_RESULTS_DIR = PROJECT_DIR / "reports" / "results" / experiment_name
REPORTS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for filename in ["results.csv", "results.png"]:
    src_file = experiment_dir / filename
    if src_file.exists():
        shutil.copy(src_file, REPORTS_RESULTS_DIR / filename)
        print("Copiado:", filename)
    else:
        print("No encontrado:", src_file)

# Desde ultralytics 8.3 las curvas de detección llevan el prefijo "Box"
# (BoxPR_curve.png); se aceptan ambos nombres y se copian al nombre sin
# prefijo, que es el esquema común de reports/results/ en todos los modelos.
figuras_eval = {
    "confusion_matrix.png": ["confusion_matrix.png"],
    "confusion_matrix_normalized.png": ["confusion_matrix_normalized.png"],
    "PR_curve.png": ["BoxPR_curve.png", "PR_curve.png"],
    "F1_curve.png": ["BoxF1_curve.png", "F1_curve.png"],
    "P_curve.png": ["BoxP_curve.png", "P_curve.png"],
    "R_curve.png": ["BoxR_curve.png", "R_curve.png"],
}

for destino, candidatos in figuras_eval.items():
    src_file = next((EVAL_DIR / c for c in candidatos if (EVAL_DIR / c).exists()), None)
    if src_file is not None:
        shutil.copy(src_file, REPORTS_RESULTS_DIR / destino)
        print(f"Copiado: {destino} (desde {src_file.name})")
    else:
        print(f"No encontrado en {EVAL_DIR}: ninguno de {candidatos}")

with open(REPORTS_RESULTS_DIR / "experiment_config_used.yaml", "w", encoding="utf-8") as file:
    yaml.safe_dump(experiment_config, file, sort_keys=False, allow_unicode=True)

print("Config usada guardada.")

Copiado: results.csv
Copiado: results.png
Copiado: confusion_matrix.png (desde confusion_matrix.png)
Copiado: confusion_matrix_normalized.png (desde confusion_matrix_normalized.png)
Copiado: PR_curve.png (desde BoxPR_curve.png)
Copiado: F1_curve.png (desde BoxF1_curve.png)
Copiado: P_curve.png (desde BoxP_curve.png)
Copiado: R_curve.png (desde BoxR_curve.png)
Config usada guardada.


In [19]:
# ============================================================
# Exportar metrics_summary.csv con el esquema común
# ============================================================
# La construcción de la fila vive en src/ y tiene tests: la lógica de resolver
# ap50/ap por ap_class_index es idéntica a la del notebook 02 y duplicarla ya
# costó una corrección doble.

from src.reporting.experiment_report import build_ultralytics_metrics_row
from src.reporting.summary import write_metrics_summary

resumen = build_ultralytics_metrics_row(
    config=experiment_config,
    box_metrics=metrics.box,
    speed=metrics.speed,
    params_M=sum(p.numel() for p in model.model.parameters()) / 1e6,
    train_time_min=train_time_min,
    device_name=DEVICE_NAME,
)

df_resumen = write_metrics_summary(REPORTS_RESULTS_DIR / "metrics_summary.csv", resumen)
display(df_resumen.T.rename(columns={0: "valor"}))

,valor
experiment,rtdetr_l
family,Transformer
model,rtdetr-l.pt
params_M,31.99
epochs,30
imgsz,640
batch,8
train_time_min,562.91
mAP50,0.7509
mAP50_95,0.415


In [20]:
# ============================================================
# Publicación de resultados
# ============================================================
# En Colab y local se commitea directo al repo. En Kaggle no hay credenciales de
# git, así que se empaqueta todo y el commit se hace después desde la máquina
# donde sí están configuradas.

import subprocess

if IN_KAGGLE:
    entregable = Path("/kaggle/working") / f"{experiment_name}_resultados"
    if entregable.exists():
        shutil.rmtree(entregable)
    shutil.copytree(REPORTS_RESULTS_DIR, entregable / "reports_results")

    # best.pt va aparte del reporte: lo necesitan el notebook 05 y la demo, y
    # pesa demasiado para commitearlo al repo.
    if best_weights.exists():
        shutil.copy(best_weights, entregable / "best.pt")

    zip_path = shutil.make_archive(str(entregable), "zip", root_dir=entregable)
    shutil.rmtree(entregable)

    print("Paquete listo:", zip_path)
    print(f"Tamaño: {Path(zip_path).stat().st_size / 1e6:.1f} MB")
    print()
    print("Pasos siguientes:")
    print("  1. Save Version > Quick Save, para conservar la corrida completa.")
    print("  2. Bajar el zip desde el panel Output.")
    print(f"  3. Descomprimir reports_results/ en "
          f"reports/results/{experiment_name}/ del repo y commitear.")
    print(f"  4. Subir best.pt a Drive en VCII_DFire/runs/{experiment_name}/"
          f"weights/ para seguir en Colab con el notebook 05.")
else:
    %cd {PROJECT_DIR}

    !git config user.name "Gabriela-Sol"
    !git config user.email "solgab.salazar@gmail.com"

    pull_result = subprocess.run(
        ["git", "pull", "--rebase", "origin", REPO_BRANCH], text=True, capture_output=True
    )
    print(pull_result.stdout, pull_result.stderr)

    if pull_result.returncode != 0:
        raise RuntimeError("No se pudo completar git pull --rebase. Revisar conflictos.")

    for path in [f"reports/results/{experiment_name}/", "configs/experiments/rtdetr_l.yaml"]:
        if Path(path).exists():
            subprocess.run(["git", "add", path], check=True)
            print("Agregado:", path)

    status = subprocess.run(["git", "status", "--short"], text=True, capture_output=True)
    print(status.stdout)

    if not status.stdout.strip():
        print("No hay cambios nuevos para commitear.")
    else:
        subprocess.run(
            ["git", "commit", "-m", f"results: update {experiment_name} outputs"], check=True
        )
        print(f"Commit creado. Para publicarlo: !git push origin {REPO_BRANCH}")

Paquete listo: /kaggle/working/rtdetr_l_resultados.zip
Tamaño: 61.9 MB

Pasos siguientes:
  1. Save Version > Quick Save, para conservar la corrida completa.
  2. Bajar el zip desde el panel Output.
  3. Descomprimir reports_results/ en reports/results/rtdetr_l/ del repo y commitear.
  4. Subir best.pt a Drive en VCII_DFire/runs/rtdetr_l/weights/ para seguir en Colab con el notebook 05.
